<img src = "imgs/tonic-logo-black-bg.webp" width="10%">

In [1]:
from tonic.transforms import ToFrame
from torch.utils.data import DataLoader
from tonic.datasets.nmnist import NMNIST
import matplotlib.pyplot as plt
n_time_steps = 10
batch_size = 32
root_dir = '/home/mazdak/projects/univ-lille/datasets'    # change it to your data path

to_raster = ToFrame(sensor_size=NMNIST.sensor_size, n_time_bins=n_time_steps)

train_dataset = NMNIST(save_to=root_dir, train=True, transform=to_raster)
test_dataset = NMNIST(save_to=root_dir, train=False, transform=to_raster)

train_dataloader = DataLoader( train_dataset, batch_size=batch_size, drop_last=True, shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=batch_size, drop_last=True,shuffle=False)

/home/mazdak/projects/univ-lille/datasets/NMNIST/train.zip
/home/mazdak/projects/univ-lille/datasets/NMNIST/test.zip


# Design Model/ Train/ Test
<img src = "imgs/sinabs-logo-lowercase-whitebg.webp" width="20%">

In [2]:
import torch.nn as nn
import sinabs.layers as sl
model = nn.Sequential(

        nn.Conv2d(2, 1, 3, bias=False),
        nn.ReLU(),    
        nn.AvgPool2d(2,2),
        nn.Linear(16*16*1, 10, bias=False),
        nn.ReLU()

)

In [3]:
model

Sequential(
  (0): Conv2d(2, 1, kernel_size=(3, 3), stride=(1, 1), bias=False)
  (1): ReLU()
  (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (3): Linear(in_features=256, out_features=10, bias=False)
  (4): ReLU()
)

In [4]:
from sinabs.from_torch import from_model
sinabs_model = from_model(model, batch_size=16)
sinabs_model

Network(
  (spiking_model): Sequential(
    (0): Conv2d(2, 1, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (1): IAFSqueeze(spike_threshold=1.0, min_v_mem=-1.0, batch_size=16, num_timesteps=-1)
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Linear(in_features=256, out_features=10, bias=False)
    (4): IAFSqueeze(spike_threshold=1.0, min_v_mem=-1.0, batch_size=16, num_timesteps=-1)
  )
  (analog_model): Sequential(
    (0): Conv2d(2, 1, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Linear(in_features=256, out_features=10, bias=False)
    (4): ReLU()
  )
)

### Quantization and Mapping using dynapcnn backend

In [5]:
from sinabs.backend.dynapcnn import DynapcnnNetwork
hw_model = DynapcnnNetwork(sinabs_model, input_shape=(2, 34, 34), dvs_input=False)
hw_model

DynapcnnNetwork(
  (sequence): Sequential(
    (0): DynapcnnLayer(
      (conv_layer): Conv2d(2, 1, kernel_size=(3, 3), stride=(1, 1), bias=False)
      (spk_layer): IAFSqueeze(spike_threshold=578.0, min_v_mem=-578.0, batch_size=16, num_timesteps=-1)
      (pool_layer): SumPool2d(norm_type=1, kernel_size=(2, 2), stride=None, ceil_mode=False)
    )
    (1): DynapcnnLayer(
      (conv_layer): Conv2d(1, 10, kernel_size=(16, 16), stride=(1, 1), bias=False)
      (spk_layer): IAFSqueeze(spike_threshold=8130.0, min_v_mem=-8130.0, batch_size=16, num_timesteps=-1)
    )
  )
)